In [1]:
!nvidia-smi

Mon Oct 19 06:07:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import subprocess, time, os

# 1. Install zstd and pciutils (pciutils enables GPU detection during Ollama install)
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Configure Ollama for GPU & 4 concurrent workers
os.environ["OLLAMA_NUM_PARALLEL"] = "4"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# 3. Start Ollama daemon in background
ollama_proc = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # Wait for daemon to initialize

# 4. Pull the Gemma 2B model (fast and lightweight for extraction)
!ollama pull gemma2:2b

# 5. Verify Ollama is running
!ollama list

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

NAME         ID              SIZE      MODIFIED               
gemma2:2b    8ccf136fdd52    1.6 GB    Less than a second ago    


In [ ]:
# [CELL 2] Clean dependency installation
!apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0 libgomp1

# 1. Install PaddlePaddle GPU
!pip install --upgrade pip -q
!pip install paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu118/ -q

# 2. Verify Paddle on GPU
import paddle
print("Paddle compiled with CUDA:", paddle.device.is_compiled_with_cuda())
print("Active Paddle device:", paddle.device.get_device())

# 3. Upgrade torch for clean NCCL matching, then install API & PaddleOCR dependencies
!pip install --upgrade torch -q
!pip install "fastapi[standard]>=0.141.1" \
             "httpx>=0.28.0" \
             "ollama>=0.4.0" \
             "pillow>=11.0.0" \
             "pydantic-settings>=2.10.0" \
             "pymupdf>=1.26.0" \
             "paddleocr" -q

print("All dependencies successfully installed without conflict!")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Paddle compiled with CUDA: True
Active Paddle device: gpu:0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.26.0+cu130 requires torch==2.11.0, but you have torch 2.14.1 which is incompatible.
All dependencies successfully installed without conflict!


In [ ]:
import zipfile, os, sys

zip_path = '/content/invoice_reader_api.zip'
extract_dir = '/content'

# Ensure clean directory extraction
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print('Project successfully extracted to /content')
else:
    print(f'Note: {zip_path} not found. If already unzipped in /content, proceeding.')

# Add src to Python path so uae_invoice_reader is immediately importable
src_path = '/content/src'
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = f'{src_path}:' + os.environ.get('PYTHONPATH', '')

# Write /content/.env
env_content = '''OLLAMA_BASE_URL=http://127.0.0.1:11434
OLLAMA_MODEL=gemma2:2b
OLLAMA_TIMEOUT_SECONDS=120.0
SAVVY_API_KEY=kBLaCcQmnuk7yTALLOtcC3iM77Xt5q8H
LOG_LEVEL=INFO
MAX_CONCURRENT_EXTRACTIONS=4
MAX_UPLOAD_MB=15
MAX_PDF_PAGES=20
MAX_IMAGE_PIXELS=40000000
PADDLE_DEVICE=gpu
PADDLE_LANG=en
PADDLE_MODEL_CACHE_DIR=/root/.model_cache/paddle
PADDLE_ENABLE_MKLDNN=false
AUTO_RELOAD=false
'''

with open('/content/.env', 'w') as f:
    f.write(env_content)
print('.env file configured with GPU and Ollama settings!')

# Verify import
import main
print('App successfully loaded and verified!')


Project successfully extracted to /content
.env file configured with GPU and Ollama settings!
App successfully loaded and verified!


In [ ]:
# Restart Uvicorn with Colab NVIDIA driver paths
import subprocess, time, requests, os

# 1. Kill old Uvicorn process
!pkill -f uvicorn 2>/dev/null || true
time.sleep(2)

# 2. Add /usr/lib64-nvidia (where Colab keeps libcuda.so.1) to LD_LIBRARY_PATH
colab_env = os.environ.copy()
nvidia_paths = ['/usr/lib64-nvidia', '/usr/local/cuda/compat']
colab_env['LD_LIBRARY_PATH'] = ':'.join([p for p in nvidia_paths if os.path.exists(p)]) + ':' + colab_env.get('LD_LIBRARY_PATH', '')

# 3. Start Uvicorn
with open('/content/uvicorn.log', 'w') as log_f:
    subprocess.Popen(
        ['uvicorn', 'main:app', '--host', '127.0.0.1', '--port', '8000'],
        cwd='/content',
        stdout=log_f,
        stderr=log_f,
        env=colab_env
    )

time.sleep(5)

# 4. Verify health
res = requests.get('http://127.0.0.1:8000/health', timeout=5)
print('Server ready! Health status:', res.status_code, res.json())


^C
Server ready! Health status: 200 {'status': 'ok', 'model': 'gemma2:2b', 'provider': 'ollama', 'device': 'gpu'}


In [ ]:
import subprocess, time, re

# 1. Download and install cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

# 2. Terminate any previous tunnel
!pkill -f cloudflared 2>/dev/null || true
time.sleep(1)

# 3. Launch cloudflared tunnel
cf_log = "/content/cloudflared.log"
with open(cf_log, "w") as log_f:
    subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=log_f,
        stderr=log_f
    )

print("Waiting for Cloudflare Tunnel URL...")
time.sleep(6)

# 4. Extract public trycloudflare.com URL from log
tunnel_url = None
with open(cf_log, "r") as log_f:
    content = log_f.read()
    matches = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
    if matches:
        tunnel_url = matches[0]

if tunnel_url:
    print("\n=======================================================")
    print("PUBLIC POSTMAN BASE URL:")
    print(tunnel_url)
    print("=======================================================")
    print(f"Health test: {tunnel_url}/health")
else:
    print("Could not find URL yet. Check log below:")
    !cat /content/cloudflared.log | grep -E "trycloudflare.com|error"

^C
Waiting for Cloudflare Tunnel URL...

PUBLIC POSTMAN BASE URL:
https://sheets-shoes-compete-extended.trycloudflare.com
Health test: https://sheets-shoes-compete-extended.trycloudflare.com/health


In [4]:
import fitz  # PyMuPDF

def create_pdf(filename, text_lines):
    doc = fitz.open()
    page = doc.new_page(width=595, height=842)  # A4 size
    y = 50
    for line in text_lines:
        page.insert_text((50, y), line, fontsize=11)
        y += 24
    doc.save(filename)
    doc.close()
    print(f"Created: {filename}")

# 1. India Tax Invoice
create_pdf("/content/india_sample.pdf", [
    "TAX INVOICE",
    "Supplier: CloudTech Solutions India Pvt Ltd",
    "Supplier GSTIN: 29AAACC1234D1Z5    PAN: AAACC1234D",
    "Supplier Address: Bangalore, Karnataka, 560001",
    "Customer: Mumbai Enterprises Ltd",
    "Customer GSTIN: 27AABCM5678E1Z2    PAN: AABCM5678E",
    "Invoice Number: INV-IN-2026-0045",
    "Invoice Date: 2026-03-15",
    "Item 1: Cloud Hosting Services | HSN: 998315 | Qty: 1 | Rate: 50000.00 | Amount: 50000.00",
    "Taxable Amount: 50000.00",
    "CGST (9%): 4500.00",
    "SGST (9%): 4500.00",
    "IGST (0%): 0.00",
    "Invoice Grand Total: 59000.00 INR"
])

# 2. USA Commercial Invoice
create_pdf("/content/usa_sample.pdf", [
    "COMMERCIAL INVOICE",
    "Vendor: Apex Industrial Supplies LLC",
    "Vendor EIN: 12-3456789",
    "Address: 742 Evergreen Terrace, Springfield, OR 97477",
    "Customer: Pacific Coast Tech Inc",
    "Customer Tax ID: 98-7654321",
    "Invoice No: USA-2026-8812",
    "Invoice Date: 2026-02-28",
    "PO Number: PO-99412",
    "Item 1: High-Speed Networking Cables | Qty: 10 | Unit Price: 45.00 | Total: 450.00",
    "Net Amount: 450.00",
    "Sales Tax Rate: 8.25%",
    "Sales Tax: 37.13",
    "Shipping & Freight: 15.00",
    "Grand Total: 502.13 USD",
    "Total Balance Payable: 502.13 USD"
])

# 3. UAE Tax Invoice
create_pdf("/content/uae_sample.pdf", [
    "TAX INVOICE",
    "Supplier Name: Gulf Trading LLC",
    "Supplier TRN: 100234567800003",
    "Supplier Address: Sheikh Zayed Road, Dubai, UAE",
    "Buyer Name: Emirates Tech Solutions",
    "Buyer TRN: 100987654300003",
    "Invoice Number: UAE-INV-9901",
    "Invoice Date: 2026-01-20",
    "Item: Ergonomic Office Chairs | Qty: 4 | Unit Price: 500.00 | Net: 2000.00 | VAT: 100.00",
    "Total Net Amount: 2000.00 AED",
    "VAT Rate: 5%",
    "Total VAT Amount: 100.00 AED",
    "Invoice Grand Total: 2100.00 AED",
    "Currency: AED"
])


ModuleNotFoundError: No module named 'fitz'

In [ ]:
import requests, json, os

BASE_URL = 'http://127.0.0.1:8000'
HEADERS = {'SAVVY_API_KEY': 'kBLaCcQmnuk7yTALLOtcC3iM77Xt5q8H'}

def test_invoice(country_name, endpoint, file_path, custom_prompt=None):
    print(f'\n=======================================================')
    print(f'TESTING {country_name}: {endpoint}')
    print(f'=======================================================')
    with open(file_path, 'rb') as f:
        files = {'file': (os.path.basename(file_path), f, 'application/pdf')}
        data = {'prompt': custom_prompt} if custom_prompt else {}
        resp = requests.post(f'{BASE_URL}{endpoint}', headers=HEADERS, files=files, data=data, timeout=90)

    print(f'Status Code: {resp.status_code}')
    try:
        result = resp.json()
        print('Success:', result.get('success'))
        if result.get('success'):
            print('Extracted Data:')
            print(json.dumps(result.get('data'), indent=2))
        else:
            print('Error Detail:')
            print(json.dumps(result.get('error'), indent=2))
    except Exception:
        print(resp.text)

# 1. Test India
test_invoice(
    country_name='INDIA',
    endpoint='/api/v1/india/invoices/extract',
    file_path='/content/india_sample.pdf',
    custom_prompt='Ensure GSTIN, CGST, SGST, and HSN codes are accurately captured.'
)

# 2. Test USA
test_invoice(
    country_name='USA',
    endpoint='/api/v1/usa/invoices/extract',
    file_path='/content/usa_sample.pdf',
    custom_prompt='Check vendor EIN and sales tax rate percentage.'
)

# 3. Test UAE
test_invoice(
    country_name='UAE',
    endpoint='/api/v1/invoices/extract',
    file_path='/content/uae_sample.pdf',
    custom_prompt='Verify 15-digit TRN numbers and 5% VAT rate.'
)



TESTING INDIA: /api/v1/india/invoices/extract
Status Code: 422
Success: False
Error Detail:
{
  "code": "document_extraction_failed",
  "detail": "Failed to render and extract PDF text with PaddleOCR: PaddleOCR could not be initialized: Unknown argument: use_gpu",
  "source": "docker_ocr",
  "retryable": false
}

TESTING USA: /api/v1/usa/invoices/extract
Status Code: 422
Success: False
Error Detail:
{
  "code": "document_extraction_failed",
  "detail": "Failed to render and extract PDF text with PaddleOCR: PaddleOCR could not be initialized: Unknown argument: use_gpu",
  "source": "docker_ocr",
  "retryable": false
}

TESTING UAE: /api/v1/invoices/extract
Status Code: 422
Success: False
Error Detail:
{
  "code": "document_extraction_failed",
  "detail": "Failed to render and extract PDF text with PaddleOCR: PaddleOCR could not be initialized: Unknown argument: use_gpu",
  "source": "docker_ocr",
  "retryable": false
}


In [ ]:
# Run this diagnostic in Colab:
!echo "=== What is inside /content/.env ==="
!grep "SAVVY_API_KEY" /content/.env

# Check what the running Python process reads:
from uae_invoice_reader.common.config import get_settings
s = get_settings()
print("\n=== What Python settings currently reads ===")
print("Key in settings:", repr(s.savvy_api_key.get_secret_value() if s.savvy_api_key else None))


In [ ]:
import os, shutil, stat

target_dir = "/content"

# Helper to force delete read-only files if any
def remove_readonly(func, path, excinfo):
    os.chmod(path, stat.S_IWRITE)
    func(path)

print(f"Deleting all files, folders, and subfolders in {target_dir}...\n")

for item in os.listdir(target_dir):
    # Optional: Skip Colab default system folders if you want to keep them
    # If you want to delete EVERYTHING including sample_data, remove this if condition:
    if item in [".config"]:
        continue

    item_path = os.path.join(target_dir, item)
    try:
        if os.path.isdir(item_path):
            # Recursively deletes the folder and all its subfolders and files
            shutil.rmtree(item_path, onerror=remove_readonly)
            print(f"Deleted folder & subfolders: {item}/")
        else:
            # Deletes individual file or symlink
            os.remove(item_path)
            print(f"Deleted file: {item}")
    except Exception as e:
        print(f"Error deleting {item}: {e}")

print("\nFinished! Remaining in /content:")
print(os.listdir(target_dir))
